# 📸 내 사진으로 데이터셋 만들고 YOLO26 학습하기

<a href="https://colab.research.google.com/github/kang-teacher/YOLO/blob/main/colab/yolo26_custom_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Roboflow 같은 외부 사이트 없이 **코랩 안에서** 전부 해요.

| 단계 | 하는 일 |
| --- | --- |
| 0 | GPU 켜기, 라이브러리 설치 |
| 1 | 설정 (찾을 물체 이름 정하기, 구글 드라이브 연결) |
| 2 | 사진 올리기 (업로드 / 웹캠으로 찍기 / 드라이브 폴더) |
| 3 | **사진마다 마우스로 바운딩 박스 그리기** |
| 4 | 라벨 확인 → 학습용 데이터셋 만들기 |
| 5 | YOLO26 학습 |
| 6 | 결과 보기, 새 사진으로 예측 |
| 7 | `best.pt` 내려받아 파이참에서 웹캠으로 쓰기 |

**시작 전에:** 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 고르세요.
위에서부터 순서대로 셀 왼쪽의 ▶ 버튼을 누르면 됩니다.

> 💡 라벨(박스)은 구글 드라이브에 바로바로 저장돼요. 코랩 연결이 끊겨도 **0번과 1번 셀만 다시 실행**하면 그리던 곳부터 이어서 할 수 있어요.

## 0. GPU 확인과 설치

In [ ]:
!nvidia-smi
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# 최신 ultralytics 설치 (YOLO26 은 8.4 버전부터 들어 있어요)
!pip install -U ultralytics pillow-heif -q

from IPython import display
display.clear_output()

# ultralytics 가 사용 기록을 보내지 않게 하기
!yolo settings sync=False

import ultralytics
ultralytics.checks()

from ultralytics import YOLO

## 1. 설정

- `CLASSES` : 찾고 싶은 물체 이름. **영어로** 써 주세요 (결과 그림에서 한글이 깨질 수 있어요).
  여러 개면 `['cup', 'phone']` 처럼 쓰고, 박스를 그릴 때 숫자키 1, 2 로 고를 수 있어요.
- `PROJECT_NAME` : 내 데이터셋 이름 (영어, 띄어쓰기 없이)
- `USE_DRIVE` : 체크하면 사진과 라벨이 구글 드라이브 `내 드라이브/yolo_datasets/` 에 저장돼요. **꼭 켜 두는 걸 추천해요.**

⚠️ 박스를 그리기 시작한 뒤에 `CLASSES` 의 **순서**를 바꾸면 라벨 번호가 꼬여요. 새 물체는 맨 뒤에 추가하세요.

In [ ]:
#@title 1. 설정 (값을 바꾼 뒤 ▶ 실행)
CLASSES = ['cup']          # 찾을 물체 이름 (영어)
PROJECT_NAME = 'my_cup'    #@param {type:"string"}
USE_DRIVE = True           #@param {type:"boolean"}

import re, json, shutil
from pathlib import Path

PROJECT_NAME = re.sub(r'[^0-9A-Za-z_\-]', '_', PROJECT_NAME.strip()) or 'my_dataset'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/yolo_datasets') / PROJECT_NAME
else:
    DATA_ROOT = Path('/content/yolo_datasets') / PROJECT_NAME

IMG_DIR = DATA_ROOT / 'images'   # 원본 사진 (한 변 최대 1280px 로 줄여서 저장)
LBL_DIR = DATA_ROOT / 'labels'   # 사진마다 같은 이름의 .txt (YOLO 형식)
IMG_DIR.mkdir(parents=True, exist_ok=True)
LBL_DIR.mkdir(parents=True, exist_ok=True)
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.heic', '.heif'}

classes_file = DATA_ROOT / 'classes.txt'
if classes_file.exists():
    old = [c for c in classes_file.read_text(encoding='utf-8').splitlines() if c.strip()]
    if old != CLASSES[:len(old)]:
        print('⚠️ 전에 쓰던 클래스와 순서가 달라요!', old, '→', CLASSES)
        print('   이미 그린 박스의 번호가 다른 물체를 가리키게 됩니다. 일부러 바꾼 게 아니면 CLASSES 를 되돌리세요.')
classes_file.write_text('\n'.join(CLASSES) + '\n', encoding='utf-8')

def image_list():
    return sorted(p for p in IMG_DIR.iterdir() if p.suffix.lower() in IMG_EXTS)

def label_path(img):
    return LBL_DIR / (img.stem + '.txt')

n_img = len(image_list())
n_lbl = sum(label_path(p).exists() for p in image_list())
print(f'📁 저장 위치: {DATA_ROOT}')
print(f'🏷️ 클래스: {CLASSES}')
print(f'🖼️ 사진 {n_img}장, 박스 그린 사진 {n_lbl}장')

## 2. 사진 올리기

세 가지 방법 중 편한 걸 쓰세요. 섞어서 써도 돼요.

- **2-A 업로드** : 내 컴퓨터의 사진을 여러 장 한꺼번에 고르거나, 사진을 묶은 `.zip` 파일을 올려요.
  파이참 라벨링 도구로 만든 zip(사진 + 라벨)을 올리면 박스까지 같이 들어와요.
- **2-B 웹캠으로 찍기** : 노트북 카메라로 물체를 여러 각도에서 찍어요.
- **2-C 구글 드라이브 폴더** : 휴대폰 사진을 드라이브에 올려 두었다면 폴더 이름만 적으면 돼요.

📌 좋은 데이터셋 팁: 물체 하나당 **30장 이상**, 각도·거리·배경·조명을 다양하게!

In [ ]:
#@title 사진 저장 도우미 (먼저 한 번 실행)
import io, zipfile
from PIL import Image, ImageOps
try:
    import pillow_heif
    pillow_heif.register_heif_opener()   # 아이폰 HEIC 사진도 열 수 있게
except ImportError:
    pass

MAX_SIDE = 1280

def _unique_stem(stem):
    stem = re.sub(r'[^\w\-]', '_', stem) or 'img'
    cand, i = stem, 1
    while (IMG_DIR / f'{cand}.jpg').exists():
        cand = f'{stem}_{i}'
        i += 1
    return cand

def add_image(data, name):
    """사진 바이트를 받아 회전 보정 + 크기 줄이기 + jpg 로 저장. 저장된 파일 이름(stem)을 돌려줌."""
    try:
        im = Image.open(io.BytesIO(data))
        im = ImageOps.exif_transpose(im).convert('RGB')   # 휴대폰 사진이 옆으로 눕는 문제 해결
    except Exception as e:
        print(f'  ✗ {name}: 사진이 아니에요 ({e.__class__.__name__})')
        return None
    im.thumbnail((MAX_SIDE, MAX_SIDE))
    stem = _unique_stem(Path(name).stem)
    im.save(IMG_DIR / f'{stem}.jpg', quality=92)
    return stem

def import_zip(data):
    """zip 안의 사진(그리고 같은 이름의 YOLO 라벨 .txt)을 가져오기"""
    zf = zipfile.ZipFile(io.BytesIO(data))
    members = [m for m in zf.namelist() if not m.startswith('__MACOSX') and not m.endswith('/')]
    txts = {Path(m).stem: m for m in members if m.lower().endswith('.txt') and Path(m).name != 'classes.txt'}
    n_img = n_lbl = 0
    for m in members:
        if Path(m).suffix.lower() in IMG_EXTS:
            stem = add_image(zf.read(m), Path(m).name)
            if stem is None:
                continue
            n_img += 1
            if Path(m).stem in txts:
                label_path(IMG_DIR / f'{stem}.jpg').write_bytes(zf.read(txts[Path(m).stem]))
                n_lbl += 1
    for m in members:
        if Path(m).name == 'classes.txt':
            zc = [c for c in zf.read(m).decode('utf-8').splitlines() if c.strip()]
            if zc != CLASSES:
                print(f'  ⚠️ zip 의 classes.txt {zc} 와 지금 CLASSES {CLASSES} 가 달라요. 1번 셀의 CLASSES 를 맞춰 주세요.')
    print(f'  📦 zip 에서 사진 {n_img}장, 라벨 {n_lbl}개 가져옴')

print('준비 완료! 이제 2-A, 2-B, 2-C 중 하나를 실행하세요.')

In [ ]:
#@title 2-A. 내 컴퓨터에서 사진 / zip 업로드
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    if name.lower().endswith('.zip'):
        import_zip(data)
    else:
        add_image(data, name)
    if os.path.exists(name):       # files.upload() 가 현재 폴더에도 남기는 사본 지우기
        os.remove(name)
print(f'✅ 지금까지 사진 {len(image_list())}장')

In [ ]:
#@title 2-B. 웹캠으로 사진 찍기 (찍기 버튼 또는 스페이스바)
from IPython.display import HTML, JSON, display
from google.colab import output
import base64

def _cam_save(data_url):
    raw = base64.b64decode(data_url.split(',', 1)[1])
    add_image(raw, f'cam_{len(image_list()):04d}.jpg')
    return JSON({'total': len(image_list())})

output.register_callback('yolo_cam_save', _cam_save)

display(HTML('''
<div style="font-family:sans-serif">
  <video id="cam" autoplay playsinline style="max-width:640px;width:100%;border-radius:8px;background:#000"></video><br>
  <button id="shot" style="font-size:18px;padding:8px 20px;margin:6px 4px 0 0">📸 찍기 (Space)</button>
  <button id="stop" style="font-size:16px;padding:8px 16px">카메라 끄기</button>
  <span id="cnt" style="margin-left:10px;font-size:16px"></span>
</div>
<script>
(async () => {
  const v = document.getElementById('cam'), cnt = document.getElementById('cnt');
  let stream;
  try {
    stream = await navigator.mediaDevices.getUserMedia({video: {width: 1280, height: 720}});
  } catch (e) {
    cnt.textContent = '카메라를 켤 수 없어요: ' + e.name + ' (브라우저 주소창 옆 카메라 권한을 허용하세요)';
    return;
  }
  v.srcObject = stream;
  async function shot() {
    const c = document.createElement('canvas');
    c.width = v.videoWidth; c.height = v.videoHeight;
    c.getContext('2d').drawImage(v, 0, 0);
    v.style.opacity = 0.3; setTimeout(() => v.style.opacity = 1, 120);
    const r = await google.colab.kernel.invokeFunction('yolo_cam_save', [c.toDataURL('image/jpeg', 0.92)], {});
    cnt.textContent = '저장됨! 지금까지 사진 ' + r.data['application/json'].total + '장';
  }
  document.getElementById('shot').onclick = shot;
  document.addEventListener('keydown', e => { if (e.code === 'Space') { e.preventDefault(); shot(); } });
  document.getElementById('stop').onclick = () => { stream.getTracks().forEach(t => t.stop()); cnt.textContent = '카메라 꺼짐'; };
})();
</script>
'''))

In [ ]:
#@title 2-C. 구글 드라이브 폴더에서 가져오기 (USE_DRIVE 가 켜져 있어야 해요)
DRIVE_FOLDER = 'my_photos'   #@param {type:"string"}
src = Path('/content/drive/MyDrive') / DRIVE_FOLDER
if not src.exists():
    print(f'✗ {src} 폴더가 없어요. 드라이브의 "내 드라이브" 바로 아래 폴더 이름을 적어 주세요.')
else:
    n = 0
    for p in sorted(src.rglob('*')):
        if p.suffix.lower() in IMG_EXTS:
            n += add_image(p.read_bytes(), p.name) is not None
        elif p.suffix.lower() == '.zip':
            import_zip(p.read_bytes())
    print(f'✅ {n}장 가져옴, 지금까지 사진 {len(image_list())}장')

## 3. 바운딩 박스 그리기 ✏️

셀을 실행하면 **아직 박스를 안 그린 첫 사진**이 나와요.

| 동작 | 마우스 / 키 |
| --- | --- |
| 박스 그리기 | 물체의 왼쪽 위에서 오른쪽 아래로 **드래그** |
| 박스 하나 지우기 | 박스 안 **더블클릭**, 또는 `Z` (마지막 박스) |
| 저장하고 다음 사진 | `Enter` 또는 스페이스바 |
| 이전 / 다음 (저장 안 함) | `A` / `D` (또는 ← →) |
| 클래스 바꾸기 | 숫자키 `1`, `2`, … |

- 박스는 물체에 **딱 맞게**, 가려진 부분은 있다고 생각하고 그려요.
- 사진에 물체가 여러 개면 **모두** 그려요. 하나라도 빼먹으면 "이건 컵이 아니야"라고 가르치는 셈이에요.
- 흔들리거나 너무 이상한 사진은 **🗑 이 사진 빼기**.
- 화면 크기가 이상하면 셀을 다시 실행하면 돼요. 저장한 박스는 그대로 남아 있어요.

In [ ]:
#@title 3. 바운딩 박스 그리기 ✏️ (실행하면 아래에 그리기 화면이 나와요)
import base64
from IPython.display import HTML, JSON, display
from google.colab import output

def _read_boxes(img):
    boxes = []
    lp = label_path(img)
    if lp.exists():
        for line in lp.read_text().splitlines():
            parts = line.split()
            if len(parts) == 5:
                boxes.append(dict(zip('cxywh', [int(parts[0])] + [float(v) for v in parts[1:]])))
    return boxes

def _get(idx):
    imgs = image_list()
    if not imgs:
        return JSON({'total': 0})
    idx = int(idx)
    if idx < 0:   # -1 이면 아직 박스를 안 그린 첫 사진
        idx = next((i for i, p in enumerate(imgs) if not label_path(p).exists()), len(imgs) - 1)
    idx = max(0, min(idx, len(imgs) - 1))
    img = imgs[idx]
    return JSON({
        'idx': idx, 'total': len(imgs), 'name': img.name,
        'src': 'data:image/jpeg;base64,' + base64.b64encode(img.read_bytes()).decode(),
        'boxes': _read_boxes(img), 'labeled': label_path(img).exists(),
        'done': sum(label_path(p).exists() for p in imgs),
    })

def _save(name, boxes):
    img = IMG_DIR / name
    lines = []
    for b in boxes:
        x, y, w, h = (min(max(float(b[k]), 0.0), 1.0) for k in 'xywh')
        if w > 0.002 and h > 0.002:
            lines.append(f"{int(b['c'])} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    label_path(img).write_text('\n'.join(lines) + ('\n' if lines else ''))
    return JSON({'ok': True, 'n': len(lines)})

def _delete(name):
    img = IMG_DIR / name
    img.unlink(missing_ok=True)
    label_path(img).unlink(missing_ok=True)
    return JSON({'ok': True})

output.register_callback('yolo_get', _get)
output.register_callback('yolo_save', _save)
output.register_callback('yolo_delete', _delete)

LABELER_HTML = r'''
<div id="lb">
  <div id="lb-bar"><div id="lb-fill"></div></div>
  <div id="lb-status">불러오는 중…</div>
  <div id="lb-classes"></div>
  <canvas id="lb-canvas" tabindex="0"></canvas>
  <div id="lb-btns">
    <button id="b-prev">◀ 이전 (A)</button>
    <button id="b-undo">↶ 마지막 박스 지우기 (Z)</button>
    <button id="b-clear">모두 지우기 (C)</button>
    <button id="b-save" class="primary">💾 저장하고 다음 ▶ (Enter)</button>
    <button id="b-next">저장 안 하고 다음 (D)</button>
    <button id="b-todo">⏭ 안 한 사진으로</button>
    <button id="b-del" class="danger">🗑 이 사진 빼기</button>
  </div>
  <div id="lb-msg"></div>
  <div id="lb-help">마우스로 <b>드래그</b>해서 물체를 꼭 맞게 감싸세요. 박스 안을 <b>더블클릭</b>하면 그 박스만 지워져요.
    물체가 여러 개면 모두 그리고, 클래스가 여러 개면 숫자키(1, 2, …)로 바꿔 가며 그리세요.</div>
</div>
<style>
  #lb { font-family: sans-serif; color: #222; max-width: 960px; }
  #lb-bar { height: 8px; background: #e5e7eb; border-radius: 4px; overflow: hidden; }
  #lb-fill { height: 100%; width: 0; background: #22c55e; transition: width .3s; }
  #lb-status { margin: 6px 0; font-size: 15px; }
  #lb-classes button { margin: 0 6px 6px 0; padding: 4px 12px; border-radius: 14px; border: 2px solid #ccc; background: #fff; cursor: pointer; font-size: 14px; }
  #lb-classes button.on { color: #fff; }
  #lb-canvas { display: block; cursor: crosshair; touch-action: none; border-radius: 6px; background: #111; outline: none; user-select: none; }
  #lb-btns { margin-top: 8px; display: flex; flex-wrap: wrap; gap: 6px; }
  #lb-btns button { padding: 8px 12px; font-size: 14px; border-radius: 6px; border: 1px solid #bbb; background: #f8f8f8; cursor: pointer; }
  #lb-btns button.primary { background: #2563eb; color: #fff; border-color: #2563eb; font-weight: bold; }
  #lb-btns button.danger { color: #b91c1c; }
  #lb-msg { min-height: 22px; margin-top: 6px; font-size: 15px; font-weight: bold; }
  #lb-help { font-size: 13px; color: #666; }
  @media (prefers-color-scheme: dark) { #lb { color: #eee; } #lb-help { color: #aaa; } }
</style>
<script>
(() => {
  const CLASSES = __CLASSES__;
  const COLORS = ['#ef4444', '#3b82f6', '#22c55e', '#f59e0b', '#a855f7', '#ec4899', '#14b8a6', '#f97316', '#84cc16', '#6366f1'];
  const $ = id => document.getElementById(id);
  const canvas = $('lb-canvas'), ctx = canvas.getContext('2d');
  const img = new Image();
  let cur = null, boxes = [], cls = 0, drag = null, mouse = null, scale = 1, emptyOk = false, busy = false;

  async function call(name, args) {
    try {
      const r = await google.colab.kernel.invokeFunction(name, args, {});
      return r.data['application/json'];
    } catch (e) {
      msg('⚠️ 코랩과 연결이 끊겼어요. 위의 1번 셀과 이 셀을 다시 실행하세요.', '#dc2626');
      throw e;
    }
  }
  function msg(t, color) { $('lb-msg').textContent = t || ''; $('lb-msg').style.color = color || '#16a34a'; }

  function drawClasses() {
    const wrap = $('lb-classes'); wrap.innerHTML = '';
    CLASSES.forEach((name, i) => {
      const b = document.createElement('button');
      const col = COLORS[i % COLORS.length];
      b.textContent = (i + 1) + '. ' + name;
      b.style.borderColor = col;
      if (i === cls) { b.className = 'on'; b.style.background = col; }
      b.onclick = () => { cls = i; drawClasses(); canvas.focus(); };
      wrap.appendChild(b);
    });
  }

  function fit() {
    const maxW = Math.min(960, (document.body.clientWidth || 960) - 10), maxH = 620;
    scale = Math.min(maxW / img.naturalWidth, maxH / img.naturalHeight, 1.5);
    canvas.width = Math.round(img.naturalWidth * scale);
    canvas.height = Math.round(img.naturalHeight * scale);
  }

  function draw() {
    const W = canvas.width, H = canvas.height;
    ctx.clearRect(0, 0, W, H);
    ctx.drawImage(img, 0, 0, W, H);
    const all = boxes.slice();
    if (drag) all.push(toBox(drag.x0, drag.y0, drag.x1, drag.y1, cls));
    all.forEach(b => {
      const col = COLORS[b.c % COLORS.length];
      const x = (b.x - b.w / 2) * W, y = (b.y - b.h / 2) * H;
      ctx.lineWidth = 3; ctx.strokeStyle = col; ctx.strokeRect(x, y, b.w * W, b.h * H);
      ctx.fillStyle = col + '33'; ctx.fillRect(x, y, b.w * W, b.h * H);
      const label = CLASSES[b.c] !== undefined ? CLASSES[b.c] : ('class ' + b.c);
      ctx.font = 'bold 14px sans-serif';
      const tw = ctx.measureText(label).width + 8;
      ctx.fillStyle = col; ctx.fillRect(x, Math.max(0, y - 20), tw, 20);
      ctx.fillStyle = '#fff'; ctx.fillText(label, x + 4, Math.max(0, y - 20) + 15);
    });
    if (mouse && !drag) {   // 십자선
      ctx.setLineDash([5, 5]); ctx.lineWidth = 1; ctx.strokeStyle = '#ffffffcc';
      ctx.beginPath(); ctx.moveTo(mouse.x, 0); ctx.lineTo(mouse.x, H); ctx.moveTo(0, mouse.y); ctx.lineTo(W, mouse.y); ctx.stroke();
      ctx.setLineDash([]);
    }
  }

  function toBox(x0, y0, x1, y1, c) {
    const W = canvas.width, H = canvas.height;
    const l = Math.min(x0, x1), r = Math.max(x0, x1), t = Math.min(y0, y1), b = Math.max(y0, y1);
    return { c: c, x: (l + r) / 2 / W, y: (t + b) / 2 / H, w: (r - l) / W, h: (b - t) / H };
  }
  function pos(e) {
    const r = canvas.getBoundingClientRect();
    return {
      x: Math.min(Math.max((e.clientX - r.left) * canvas.width / r.width, 0), canvas.width),
      y: Math.min(Math.max((e.clientY - r.top) * canvas.height / r.height, 0), canvas.height),
    };
  }

  canvas.addEventListener('pointerdown', e => {
    if (!cur || e.button !== 0) return;
    canvas.setPointerCapture(e.pointerId);
    const p = pos(e); drag = { x0: p.x, y0: p.y, x1: p.x, y1: p.y };
    emptyOk = false;
  });
  canvas.addEventListener('pointermove', e => {
    mouse = pos(e);
    if (drag) { drag.x1 = mouse.x; drag.y1 = mouse.y; }
    draw();
  });
  canvas.addEventListener('pointerup', e => {
    if (!drag) return;
    const p = pos(e);
    if (Math.abs(p.x - drag.x0) > 4 && Math.abs(p.y - drag.y0) > 4) {
      boxes.push(toBox(drag.x0, drag.y0, p.x, p.y, cls));
      msg('박스 ' + boxes.length + '개 · 다 그렸으면 Enter', '#2563eb');
    }
    drag = null; draw();
  });
  canvas.addEventListener('pointerleave', () => { mouse = null; draw(); });
  canvas.addEventListener('dblclick', e => {
    const p = pos(e), W = canvas.width, H = canvas.height;
    let best = -1, area = Infinity;
    boxes.forEach((b, i) => {
      const inside = Math.abs(p.x / W - b.x) <= b.w / 2 && Math.abs(p.y / H - b.y) <= b.h / 2;
      if (inside && b.w * b.h < area) { best = i; area = b.w * b.h; }
    });
    if (best >= 0) { boxes.splice(best, 1); draw(); msg('박스를 지웠어요', '#d97706'); }
  });

  async function load(idx) {
    if (busy) return;
    busy = true;
    try {
      const d = await call('yolo_get', [idx]);
      if (!d.total) { $('lb-status').textContent = '사진이 없어요. 2번 단계에서 사진을 먼저 올리세요.'; return; }
      cur = d; boxes = d.boxes; emptyOk = false; drag = null;
      await new Promise((ok, bad) => { img.onload = ok; img.onerror = bad; img.src = d.src; });
      fit(); draw(); update();
      canvas.focus();
    } finally { busy = false; }
  }

  function update() {
    $('lb-status').innerHTML = '<b>' + (cur.idx + 1) + ' / ' + cur.total + '</b>번째 사진 · ' + cur.name +
      (cur.labeled ? ' ✅' : ' (아직 안 함)') + ' &nbsp;|&nbsp; 박스 그린 사진 <b>' + cur.done + '</b>장';
    $('lb-fill').style.width = (100 * cur.done / cur.total) + '%';
  }

  let saving = false;
  async function save() {
    if (!cur || busy || saving) return;
    if (!boxes.length && !emptyOk) {
      emptyOk = true;
      msg('박스가 없어요! 물체가 없는 사진이 맞으면 한 번 더 누르세요 (빈 사진으로 저장).', '#d97706');
      return;
    }
    saving = true;
    try { await call('yolo_save', [cur.name, boxes]); } finally { saving = false; }
    const last = cur.idx + 1 >= cur.total;
    if (last) {
      await load(cur.idx);
      msg(cur.done >= cur.total ? '🎉 모든 사진 완료! 아래 4번 셀로 넘어가세요.' : '마지막 사진이에요. "안 한 사진으로"를 눌러 남은 사진을 확인하세요.');
    } else {
      await load(cur.idx + 1);
      msg('저장했어요 ✔');
    }
  }

  let delArmed = false;
  async function del() {
    if (!cur || busy) return;
    if (!delArmed) { delArmed = true; msg('이 사진을 데이터셋에서 지울까요? 한 번 더 누르면 지워요.', '#dc2626'); setTimeout(() => delArmed = false, 4000); return; }
    delArmed = false;
    await call('yolo_delete', [cur.name]);
    await load(cur.idx);
    msg('사진을 지웠어요', '#d97706');
  }

  $('b-prev').onclick = () => cur && load(cur.idx - 1).then(() => msg(''));
  $('b-next').onclick = () => cur && load(cur.idx + 1).then(() => msg(''));
  $('b-todo').onclick = () => load(-1).then(() => msg(''));
  $('b-undo').onclick = () => { boxes.pop(); draw(); canvas.focus(); };
  $('b-clear').onclick = () => { boxes = []; draw(); canvas.focus(); };
  $('b-save').onclick = save;
  $('b-del').onclick = del;

  document.addEventListener('keydown', e => {
    if (e.ctrlKey || e.metaKey || e.altKey) return;
    const k = e.key.toLowerCase();
    if (/^[1-9]$/.test(k) && +k <= CLASSES.length) { cls = +k - 1; drawClasses(); draw(); }
    else if (k === 'enter' || k === ' ') { e.preventDefault(); save(); }
    else if (k === 'z') boxes.pop(), draw();
    else if (k === 'c') boxes = [], draw();
    else if (k === 'a' || k === 'arrowleft') $('b-prev').click();
    else if (k === 'd' || k === 'arrowright') $('b-next').click();
    else return;
    e.preventDefault();
  });

  drawClasses();
  load(-1);
})();
</script>
'''

display(HTML(LABELER_HTML.replace('__CLASSES__', json.dumps(CLASSES))))

## 4. 라벨 확인 → 학습용 데이터셋 만들기

박스를 그린 사진을 **학습용(train) 80%**, **검사용(val) 20%** 로 나눠요.
모델은 train 사진으로 공부하고, 처음 보는 val 사진으로 시험을 봐요.

In [ ]:
#@title 4-1. 박스 잘 그렸는지 확인
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

labeled = [p for p in image_list() if label_path(p).exists()]
counts = [0] * len(CLASSES)
n_empty = 0
for p in labeled:
    rows = [l.split() for l in label_path(p).read_text().splitlines() if l.strip()]
    n_empty += not rows
    for r in rows:
        c = int(r[0])
        if c < len(counts):
            counts[c] += 1
print(f'박스 그린 사진 {len(labeled)}장 (빈 사진 {n_empty}장) / 전체 {len(image_list())}장')
for name, n in zip(CLASSES, counts):
    print(f'  {name}: 박스 {n}개')

sample = random.sample(labeled, min(8, len(labeled)))
if sample:
    cols = 4
    rows_n = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(16, 4 * rows_n), squeeze=False)
    colors = ['#ef4444', '#3b82f6', '#22c55e', '#f59e0b', '#a855f7', '#ec4899', '#14b8a6', '#f97316', '#84cc16', '#6366f1']
    for ax in axes.flat:
        ax.axis('off')
    for ax, p in zip(axes.flat, sample):
        im = Image.open(p)
        W, H = im.size
        ax.imshow(im)
        ax.set_title(p.name if p.name.isascii() else '', fontsize=9)   # 한글 이름은 그림에서 깨져서 생략
        for l in label_path(p).read_text().splitlines():
            if not l.strip():
                continue
            c, x, y, w, h = l.split()
            c, x, y, w, h = int(c), float(x) * W, float(y) * H, float(w) * W, float(h) * H
            col = colors[c % len(colors)]
            ax.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h, fill=False, lw=2, ec=col))
            ax.text(x - w / 2, y - h / 2, CLASSES[c] if c < len(CLASSES) else c, color='white', backgroundcolor=col, fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
#@title 4-2. train / val 로 나누고 data.yaml 만들기
import yaml

VAL_RATIO = 0.2   #@param {type:"number"}

labeled = [p for p in image_list() if label_path(p).exists()]
if len(labeled) < 2:
    raise SystemExit('박스 그린 사진이 2장 이상 있어야 해요. 3번 단계를 먼저 하세요.')
if len(labeled) < 20:
    print(f'⚠️ 사진이 {len(labeled)}장뿐이에요. 학습은 되지만 30장 이상이면 훨씬 잘 찾아요.')

random.seed(0)
random.shuffle(labeled)
n_val = max(1, round(len(labeled) * VAL_RATIO))
splits = {'val': labeled[:n_val], 'train': labeled[n_val:]}

DATASET = Path(HOME) / 'datasets' / PROJECT_NAME   # 학습은 빠른 코랩 디스크에서
shutil.rmtree(DATASET, ignore_errors=True)
for split, items in splits.items():
    (DATASET / 'images' / split).mkdir(parents=True)
    (DATASET / 'labels' / split).mkdir(parents=True)
    for p in items:
        shutil.copy(p, DATASET / 'images' / split / p.name)
        shutil.copy(label_path(p), DATASET / 'labels' / split / label_path(p).name)

data_yaml = DATASET / 'data.yaml'
data_yaml.write_text(yaml.safe_dump({
    'path': str(DATASET),
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: n for i, n in enumerate(CLASSES)},
}, allow_unicode=True, sort_keys=False), encoding='utf-8')
(DATASET / 'classes.txt').write_text('\n'.join(CLASSES) + '\n', encoding='utf-8')

# 데이터셋을 zip 으로 백업 (파이참에서 쓰거나 친구와 나눌 때)
zip_path = shutil.make_archive(str(DATA_ROOT.parent / f'{PROJECT_NAME}_dataset'), 'zip', DATASET)

print(f'train {len(splits["train"])}장, val {len(splits["val"])}장')
print(f'📄 {data_yaml}')
print(data_yaml.read_text())
print(f'📦 백업 zip: {zip_path}')

## 5. YOLO26 학습 🚀

- `MODEL` : `yolo26n.pt` (가장 작고 빠름, 추천) / `yolo26s.pt` (조금 크고 더 정확)
- `EPOCHS` : 전체 사진을 몇 번 반복해서 공부할지. 사진 50장 기준 T4 GPU 에서 50번이 몇 분이면 끝나요.
- GPU 없이(CPU) 돌리면 아주 느려요. `EPOCHS` 를 20 정도로 줄이세요.

예전 코드의 `!yolo task=detect mode=train ...` 와 같은 일을 파이썬 코드로 해요.

In [ ]:
#@title 5. 학습 시작
import torch

MODEL = 'yolo26n.pt'   #@param ['yolo26n.pt', 'yolo26s.pt']
EPOCHS = 50            #@param {type:"integer"}
IMGSZ = 640            #@param [416, 640, 800] {type:"raw"}

if not torch.cuda.is_available():
    print('⚠️ GPU 가 없어요! 런타임 → 런타임 유형 변경 → T4 GPU 를 고르면 훨씬 빨라요.')

model = YOLO(MODEL)
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=16,
    plots=True,
    project=f'{HOME}/runs',
    name=PROJECT_NAME,
    exist_ok=True,
)
# 같은 뜻의 명령어:
# !yolo task=detect mode=train model=yolo26n.pt data={data_yaml} epochs=50 imgsz=640 plots=True

RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / 'weights' / 'best.pt'
shutil.copy(BEST, DATA_ROOT / f'{PROJECT_NAME}_best.pt')   # 드라이브에도 보관
print(f'✅ 학습 끝! 가장 좋은 모델: {BEST}')
print(f'   드라이브 사본: {DATA_ROOT / f"{PROJECT_NAME}_best.pt"}')

## 6. 결과 보기

- `results.png` : 학습이 진행될수록 loss(틀린 정도)는 내려가고 mAP(정확도)는 올라가면 잘 배운 거예요.
- `confusion_matrix.png` : 정답과 예측을 표로 비교해요.
- `val_batch0_pred.jpg` : 처음 보는 val 사진에서 모델이 찾은 결과예요.

In [ ]:
#@title 6-1. 학습 그래프와 검사 결과
from IPython.display import Image as IPImage

for f in ['results.png', 'confusion_matrix.png', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    if (RUN_DIR / f).exists():
        print(f)
        display(IPImage(filename=str(RUN_DIR / f), width=900))

In [ ]:
#@title 6-2. 새 사진으로 예측해 보기 (학습에 안 쓴 사진을 올려 보세요)
CONF = 0.25   #@param {type:"slider", min:0.05, max:0.9, step:0.05}

my_model = YOLO(str(BEST))
test = files.upload()
for name, data in test.items():
    im = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert('RGB')
    results = my_model.predict(source=im, conf=CONF)
    r = results[0]
    print(f'{name}: {len(r.boxes)}개 찾음', [f'{r.names[int(c)]} {p:.2f}' for c, p in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist())])
    display(Image.fromarray(r.plot()[:, :, ::-1]))   # plot() 은 BGR 이라 RGB 로 뒤집기
    if os.path.exists(name):
        os.remove(name)

## 7. 내 모델(best.pt) 내려받아 파이참에서 쓰기

아래 셀을 실행하면 `my_cup_best.pt` 같은 파일이 내려받아져요.
(드라이브의 `yolo_datasets/프로젝트이름/` 폴더에도 들어 있어요.)

파이참 프로젝트 폴더에 넣고, 수업 때 쓴 코드에서 **모델 이름만 바꾸면** 웹캠으로 내 물체를 찾아요.

```python
from ultralytics import YOLO
import cv2

model = YOLO('my_cup_best.pt')     # ← 내려받은 파일 이름
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, image = cap.read()
    if not success:
        continue
    results = model(image, conf=0.5)
    cv2.imshow('img', results[0].plot())
    if cv2.waitKey(1) == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()
```

> 파이참에서도 `pip install -U ultralytics` 로 최신 버전을 설치해야 YOLO26 모델을 읽을 수 있어요.

In [ ]:
#@title 7. best.pt 내려받기
dl = Path(HOME) / f'{PROJECT_NAME}_best.pt'
shutil.copy(BEST, dl)
files.download(str(dl))

## 🛠 문제가 생기면

| 증상 | 해결 |
| --- | --- |
| 3번 화면에서 버튼을 눌러도 반응이 없음 | 코랩 연결이 끊긴 거예요. **0번 설치 셀 → 1번 → 사진 저장 도우미 → 3번** 셀을 다시 실행. 박스는 드라이브에 남아 있어요. |
| `name 'image_list' is not defined` 같은 오류 | 위의 셀을 건너뛴 거예요. 1번 셀부터 순서대로 실행. |
| 키보드 단축키가 안 먹음 | 사진을 한 번 클릭한 뒤 눌러 보세요. |
| 웹캠이 안 켜짐 | 주소창 옆 카메라 아이콘에서 권한 허용, 다른 프로그램(줌 등)이 카메라를 쓰고 있지 않은지 확인. |
| GPU 를 쓸 수 없다는 메시지 | 무료 GPU 사용량을 다 쓴 거예요. CPU 로도 되지만 느리니 `EPOCHS` 를 20 으로 줄이세요. |
| 학습이 잘 안 됨 (mAP 가 낮음) | 사진을 더 모으고(30~100장), 박스를 빠짐없이 딱 맞게 그렸는지 4-1 에서 확인. |
| 코랩이 아예 안 될 때 | 저장소의 `local/` 폴더에 있는 **파이참용 라벨링 도구**로 박스를 그린 뒤, 만든 zip 을 2-A 에서 올리면 이어서 학습할 수 있어요. |